In [1]:
import gensim
from gensim.models import Word2Vec, KeyedVectors

In [2]:
import gensim.downloader as api
wv = api.load('word2vec-google-news-300')

In [3]:
import pandas as pd
messages = pd.read_csv('SpamClassifier/SMSSpamCollection.txt',sep='\t',names=['label','message'])
messages

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


#### Preprocessing

In [4]:
import nltk
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [5]:
import re
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-z]',' ',messages['message'][i])
    review = review.lower()
    review = review.split()
    review = [lemmatizer.lemmatize(word) for word in review]
    review = ' '.join(review)
    corpus.append(review)

In [6]:
#you will come back here just skip now
#this code is for missing 3 records while preprocessing
[[i,j,k] for i,j,k in zip(list(map(len,corpus)),corpus, messages['message']) if i<1]

[[0, '', '645'], [0, '', ':) '], [0, '', ':-) :-)']]

In [7]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

simple_preprocess() #Convert a document into a list of lowercase tokens, ignoring tokens that are too short or too long.

In [8]:
#you can use this block either corpus[] one - this is alternate method 
words = []
for sent in corpus:
    sent_token = sent_tokenize(sent)
    for sent in sent_token:
        words.append(simple_preprocess(sent))

In [9]:
#words

### Training model from scratch

In [10]:
model = gensim.models.Word2Vec(words)

In [11]:
#to get all the vocabulary
#model.wv.index_to_key

In [12]:
#vocabulary size
model.corpus_count

5569

In [13]:
#More number of epochs more better it is
model.epochs

5

In [14]:
model.wv.similar_by_word('free')

[('mobile', 0.997280478477478),
 ('txt', 0.9972739815711975),
 ('reply', 0.9963344931602478),
 ('text', 0.9957658648490906),
 ('stop', 0.9955301880836487),
 ('claim', 0.9947347044944763),
 ('camcorder', 0.9946491122245789),
 ('nokia', 0.9944616556167603),
 ('tone', 0.9943804740905762),
 ('call', 0.9939714074134827)]

In [15]:
model.wv['good']

array([-0.2183611 ,  0.43229958,  0.10008503,  0.18527628,  0.07508014,
       -0.7232441 ,  0.21471132,  0.72006965, -0.41436878, -0.16897716,
       -0.22502427, -0.6251931 ,  0.01139718,  0.13262786,  0.29575804,
       -0.27587402,  0.21055596, -0.35998902, -0.13652839, -0.8727445 ,
        0.35699075,  0.3237382 ,  0.12384945, -0.2536798 , -0.16325463,
        0.07613643, -0.5140913 , -0.326027  , -0.40702552,  0.0261209 ,
        0.4132509 ,  0.0101488 ,  0.14429681, -0.36781722, -0.17217514,
        0.5322103 , -0.03673089, -0.22607398, -0.16417654, -0.54902494,
        0.17105809, -0.37899518, -0.28953075,  0.04067663,  0.42564663,
       -0.00103441, -0.27626023, -0.11463377,  0.17373396,  0.20950197,
        0.23484056, -0.30301857, -0.15390384,  0.04064778, -0.13639806,
        0.30895826,  0.22071992,  0.11512545, -0.41539037,  0.13384272,
       -0.03776174,  0.16304764, -0.13088329, -0.1649782 , -0.45960978,
        0.36707166,  0.08733142,  0.27846646, -0.47204366,  0.46

## Average Word2Vec Implementation

In [16]:
def avg_word2vec(doc):
    return np.mean([model.wv[word] for word in doc if word in model.wv.index_to_key],axis=0)

In [17]:
!pip install tqdm

In [18]:
from tqdm import tqdm  #progress bar for monitoring

In [19]:
#apply for entire sentenses
import numpy as np
X = []
for i in tqdm(range(len(words))):
    X.append(avg_word2vec(words[i]))

C:\Users\Win\AppData\Roaming\Python\Python313\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\Win\AppData\Roaming\Python\Python313\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 5569/5569 [00:00<00:00, 12414.14it/s]


In [20]:
#X

In [21]:
len(X)

5569

In [22]:
#independent features
X_new = np.array(X, dtype=object)  # ✅ No error, but you lose vectorized operations

In [23]:
X_new.shape

(5569,)

In [24]:
X_new[0].shape

(100,)

In [25]:
#X_new

In [26]:
#dependent feature
y = pd.get_dummies(messages['label'])
y = y.iloc[:,0].values

In [27]:
X_new.shape , messages.shape , y.shape

((5569,), (5572, 2), (5572,))

shape is mismatching means 3 records are removed in preprocessing or cleaning - check for code in preprocessing/cleaning section(3 records missing code)

In [28]:
#to solve record mismatch issue
y_new = messages[list(map(lambda x:len(x)>0,corpus))]
y_new = pd.get_dummies(y_new['label'])
y_new = y_new.iloc[:,0].values

In [29]:
y_new = y_new.astype(int)

In [30]:
y_new

array([1, 1, 0, ..., 1, 1, 1], shape=(5569,))

In [31]:
y_new.shape

(5569,)

this matches shape with X_new

we need to add each vector values of sentense in DATAFRAME

In [32]:
X[0].reshape(1,-1).shape  # changes into rows for dataframe

(1, 100)

In [33]:
## This is final independent features
df =pd.DataFrame()
for i in range(0,len(X)):
    df = df._append(pd.DataFrame(X[i].reshape(1,-1)),ignore_index = True)

C:\Users\Win\AppData\Local\Temp\ipykernel_8200\2619653297.py:4: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = df._append(pd.DataFrame(X[i].reshape(1,-1)),ignore_index = True)


In [34]:
df

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.148785,0.288177,0.089328,0.134421,0.057298,-0.519951,0.137445,0.479138,-0.284013,-0.122553,...,0.308885,0.148286,0.010663,0.128205,0.388202,0.297137,0.100473,-0.203980,0.143550,-0.117190
1,-0.123296,0.245044,0.071176,0.116818,0.043382,-0.449003,0.109962,0.411647,-0.233793,-0.089500,...,0.263136,0.123028,0.007345,0.108057,0.328454,0.243726,0.087386,-0.177131,0.116995,-0.101242
2,-0.145997,0.310856,0.114052,0.175454,0.056285,-0.574374,0.143752,0.484165,-0.302161,-0.136303,...,0.310261,0.157816,-0.003784,0.131421,0.404208,0.284142,0.071695,-0.229220,0.174305,-0.115723
3,-0.200461,0.386343,0.112054,0.178695,0.074551,-0.687535,0.180522,0.649565,-0.382159,-0.153886,...,0.414969,0.190110,0.010366,0.177628,0.509117,0.398198,0.141318,-0.279114,0.183384,-0.148689
4,-0.180910,0.331334,0.105899,0.153910,0.081041,-0.598043,0.152555,0.567995,-0.333203,-0.142399,...,0.366724,0.159651,0.013342,0.152746,0.440565,0.348184,0.115437,-0.247302,0.161775,-0.135008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5564,-0.173341,0.364410,0.130061,0.186634,0.071297,-0.653269,0.176405,0.567386,-0.344577,-0.163657,...,0.366360,0.181997,-0.000089,0.137293,0.477701,0.338323,0.094480,-0.263740,0.197100,-0.132215
5565,-0.194230,0.344900,0.118986,0.177799,0.090689,-0.648809,0.161707,0.599171,-0.361807,-0.146503,...,0.385752,0.178188,0.001675,0.160313,0.477148,0.364552,0.110712,-0.264697,0.179979,-0.134906
5566,-0.200268,0.397584,0.113000,0.171259,0.068360,-0.692526,0.185768,0.652301,-0.382190,-0.170759,...,0.421862,0.205609,0.020551,0.178110,0.526477,0.412143,0.146504,-0.264776,0.192751,-0.164541
5567,-0.183149,0.355873,0.109644,0.167661,0.069834,-0.636075,0.164364,0.587804,-0.347617,-0.153619,...,0.376099,0.178549,0.020313,0.157330,0.475089,0.362752,0.116210,-0.250206,0.177618,-0.144750


In [35]:
df.shape

(5569, 100)

In [36]:
### Independent features
X = df

In [37]:
y_new

array([1, 1, 0, ..., 1, 1, 1], shape=(5569,))

### Train test split

In [39]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y_new,test_size=.20,random_state=40)


In [40]:
X_train.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
15,-0.152753,0.317334,0.108104,0.163126,0.061829,-0.580145,0.146019,0.505846,-0.312845,-0.138514,...,0.320490,0.167359,0.004099,0.134783,0.420909,0.306370,0.091151,-0.228724,0.173737,-0.124298
4464,-0.145211,0.307473,0.095360,0.146254,0.054308,-0.546100,0.136533,0.492503,-0.287375,-0.129515,...,0.309729,0.155606,0.008320,0.126050,0.400147,0.286717,0.083028,-0.210191,0.151863,-0.126797
3432,-0.193853,0.373291,0.111041,0.166379,0.080944,-0.653646,0.180643,0.609022,-0.357737,-0.161368,...,0.393975,0.188398,0.020410,0.163497,0.487590,0.382908,0.136930,-0.259613,0.188526,-0.142738
5270,-0.223996,0.388071,0.110237,0.163427,0.100222,-0.685908,0.172626,0.673475,-0.387927,-0.166508,...,0.437045,0.182830,0.027445,0.171457,0.518231,0.427044,0.151013,-0.280014,0.184466,-0.152482
1124,-0.226335,0.357518,0.108094,0.141249,0.115932,-0.632812,0.158145,0.634553,-0.358385,-0.170846,...,0.423962,0.158264,0.009240,0.154978,0.466072,0.413856,0.139994,-0.275699,0.166167,-0.120837


In [41]:
y_train

array([0, 1, 1, ..., 1, 1, 1], shape=(4455,))

### Creating model

In [42]:
from sklearn.ensemble import RandomForestClassifier
classifier =RandomForestClassifier()

In [43]:
classifier.fit(X_train,y_train)

RandomForestClassifier()

In [45]:
y_pred = classifier.predict(X_test)

In [44]:
from sklearn.metrics import accuracy_score,classification_report

In [46]:
print(accuracy_score(y_test,y_pred))

0.966786355475763


In [47]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.88      0.86      0.87       147
           1       0.98      0.98      0.98       967

    accuracy                           0.97      1114
   macro avg       0.93      0.92      0.93      1114
weighted avg       0.97      0.97      0.97      1114

